In [ ]:
# ================================================================
# PROJETO DE ZOONOSES
# TRATAMENTO DOS DADOS BRUTOS — SQLITE
# VERSÃO 1.4 — CONSOLIDADA E CORRIGIDA
#
# - Preserva os registros brutos e os dados manuais.
# - Trata os cinco conjuntos.
# - Consolida duplicidades exatas e parciais.
# - Mantém mapeamento persistente dos animais sintéticos.
# - Evita vínculos acidentais com animais manuais.
# - Reprocessa registros que aguardam vínculo.
# - Permite ocorrências/eventos sem identificação de animal.
# - Evita duplicação no histórico de qualidade.
# - Valida datas, vacinas e relacionamentos.
# - Atualiza status após óbito/eutanásia válidos.
# - Permite processamento incremental.
# ================================================================


# 1. IMPORTAÇÕES

import os
import json
import sqlite3

from datetime import date, datetime
from collections import Counter, defaultdict


# 2. CONFIGURAÇÕES

PASTA_PROJETO = "/content/PROJETO_ZOONOSES"

CAMINHO_BANCO = os.path.join(
    PASTA_PROJETO,
    "banco_zoonoses_v2.db"
)

VERSAO_TRATAMENTO = "1.4"

MUNICIPIOS_VALIDOS = {
    "Feijó",
    "Cruzeiro do Sul",
    "Sena Madureira",
    "Rio Branco",
    "Tarauacá"
}

ESPECIES_VALIDAS = {"Cachorro", "Gato"}
SEXOS_VALIDOS = {"Macho", "Fêmea"}
STATUS_VALIDOS = {"Ativo", "Inativo"}

MOTIVOS_ACOLHIMENTO = {
    "Maus-tratos", "Abandono", "Acidente", "Doença",
    "Recolhimento preventivo", "Entrega voluntária",
    "Animal perdido", "Resgate de filhotes", "Outros"
}

ORIGENS_ACOLHIMENTO = {
    "Resgate pela equipe", "Denúncia da população",
    "Entrega pelo tutor", "Órgão público",
    "Clínica veterinária", "Recolhimento nas ruas"
}

VACINAS_POR_ESPECIE = {
    "Cachorro": {"Antirrábica", "V8", "V10"},
    "Gato": {
        "Antirrábica",
        "Polivalente felina",
        "Leucemia felina"
    }
}

SITUACOES_VACINA = {
    "Aplicada", "Pendente", "Aguardando reforço"
}

SITUACOES_DIAGNOSTICO = {
    "Em tratamento", "Em acompanhamento", "Recuperado"
}

SITUACOES_OCORRENCIA = {
    "Em investigação", "Em acompanhamento", "Concluída"
}

TIPOS_EVENTO = {"Óbito", "Eutanásia"}

CONJUNTOS = {
    "animais",
    "vacinacoes",
    "diagnosticos",
    "maus_tratos",
    "obitos_eutanasias"
}

STATUS_TERMINAIS = {"TRATADO", "REJEITADO"}


# 3. VERIFICAÇÕES INICIAIS

if not os.path.isdir(PASTA_PROJETO):
    raise FileNotFoundError(
        f"Pasta do projeto não encontrada: {PASTA_PROJETO}"
    )

if not os.path.isfile(CAMINHO_BANCO):
    raise FileNotFoundError(
        "Banco SQLite não encontrado. Execute primeiro "
        "o notebook de construção do banco."
    )


# 4. CONEXÃO COM SQLITE

conexao = sqlite3.connect(CAMINHO_BANCO, timeout=60)
conexao.row_factory = sqlite3.Row
conexao.execute("PRAGMA foreign_keys = ON")
conexao.execute("PRAGMA busy_timeout = 60000")
cursor = conexao.cursor()


# 5. VERIFICAÇÃO DA ESTRUTURA

def tabelas_existentes():
    return {
        linha["name"]
        for linha in cursor.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
        """).fetchall()
    }


def colunas_tabela(nome):
    # O nome da tabela vem de constantes internas do próprio notebook.
    return {
        linha["name"]
        for linha in cursor.execute(
            f'PRAGMA table_info("{nome}")'
        ).fetchall()
    }


tabelas_obrigatorias = {
    "registros_brutos",
    "animais",
    "vacinacoes",
    "diagnosticos",
    "maus_tratos",
    "obitos_eutanasias",
    "auditoria",
    "lotes"
}

faltantes = tabelas_obrigatorias - tabelas_existentes()

if faltantes:
    conexao.close()
    raise RuntimeError(
        f"Tabelas obrigatórias ausentes: {sorted(faltantes)}"
    )


colunas_obrigatorias = {
    "registros_brutos": {
        "id_registro", "id_lote", "conjunto", "dados_json"
    },
    "animais": {
        "id_animal", "nome", "especie", "status", "origem"
    },
    "vacinacoes": {
        "id_animal", "nome_vacina", "situacao"
    },
    "diagnosticos": {
        "id_animal", "diagnostico"
    },
    "maus_tratos": {
        "id_animal", "tipo_ocorrencia"
    },
    "obitos_eutanasias": {
        "id_animal", "tipo_evento"
    },
    "auditoria": {
        "entidade", "operacao", "data_operacao"
    }
}

for tabela, esperadas in colunas_obrigatorias.items():
    ausentes = esperadas - colunas_tabela(tabela)

    if ausentes:
        conexao.close()
        raise RuntimeError(
            f"Colunas ausentes em {tabela}: {sorted(ausentes)}. "
            "Confira o notebook de construção do banco."
        )


# 6. TABELAS DE CONTROLE, QUALIDADE E MAPEAMENTO

cursor.execute("""
CREATE TABLE IF NOT EXISTS controle_processamento (
    id_registro_bruto INTEGER PRIMARY KEY,
    id_lote INTEGER,
    conjunto TEXT NOT NULL,
    versao_tratamento TEXT NOT NULL,
    status TEXT NOT NULL,
    id_animal_final TEXT,
    observacao TEXT,
    data_processamento TEXT NOT NULL,
    FOREIGN KEY (id_registro_bruto)
        REFERENCES registros_brutos(id_registro)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS qualidade_dados (
    id_qualidade INTEGER PRIMARY KEY AUTOINCREMENT,
    id_registro_bruto INTEGER,
    id_lote INTEGER,
    conjunto TEXT NOT NULL,
    id_animal TEXT,
    campo TEXT,
    valor_original TEXT,
    valor_corrigido TEXT,
    tipo_problema TEXT NOT NULL,
    acao TEXT NOT NULL,
    data_identificacao TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS mapeamento_animais (
    id_animal_origem TEXT PRIMARY KEY,
    id_animal_final TEXT NOT NULL,
    id_lote INTEGER,
    data_mapeamento TEXT NOT NULL
)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_qualidade_conjunto
ON qualidade_dados(conjunto)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_controle_lote
ON controle_processamento(id_lote)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_controle_status
ON controle_processamento(conjunto, status)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_mapeamento_animal_final
ON mapeamento_animais(id_animal_final)
""")

conexao.commit()


# 7. FUNÇÕES AUXILIARES

hoje = date.today()


def texto(valor):
    if valor is None:
        return ""
    return str(valor).strip()


def origem_manual(valor):
    return texto(valor).casefold() == "manual"


def data_valida(valor):
    """
    Aceita YYYY-MM-DD e valores ISO com horário.
    Retorna somente YYYY-MM-DD.
    Datas futuras ou inválidas retornam None.
    """
    valor = texto(valor)

    if not valor:
        return None

    try:
        # datetime.fromisoformat aceita data e hora ISO.
        # O recorte também protege entradas com sufixos adicionais.
        data = date.fromisoformat(valor[:10])

        if data > hoje:
            return None

        return data.isoformat()

    except (ValueError, TypeError):
        return None


def serializar(valor):
    return json.dumps(
        valor,
        ensure_ascii=False,
        sort_keys=True,
        default=str
    )


def registrar_problema(
    id_registro,
    id_lote,
    conjunto,
    id_animal,
    campo,
    original,
    corrigido,
    problema,
    acao
):
    """
    Insere o problema somente se a mesma ocorrência ainda não existir.
    Isso impede que reprocessamentos de registros pendentes
    multipliquem linhas idênticas na tabela qualidade_dados.
    """
    original_json = serializar(original)
    corrigido_json = serializar(corrigido)

    existe = cursor.execute("""
        SELECT 1
        FROM qualidade_dados
        WHERE id_registro_bruto IS ?
          AND conjunto = ?
          AND campo IS ?
          AND valor_original = ?
          AND valor_corrigido = ?
          AND tipo_problema = ?
          AND acao = ?
        LIMIT 1
    """, (
        id_registro,
        conjunto,
        campo,
        original_json,
        corrigido_json,
        problema,
        acao
    )).fetchone()

    if existe:
        return

    cursor.execute("""
        INSERT INTO qualidade_dados (
            id_registro_bruto,
            id_lote,
            conjunto,
            id_animal,
            campo,
            valor_original,
            valor_corrigido,
            tipo_problema,
            acao,
            data_identificacao
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        id_registro,
        id_lote,
        conjunto,
        id_animal,
        campo,
        original_json,
        corrigido_json,
        problema,
        acao,
        datetime.now().isoformat(timespec="seconds")
    ))


def controle_existe(id_registro):
    resultado = cursor.execute("""
        SELECT status
        FROM controle_processamento
        WHERE id_registro_bruto = ?
    """, (id_registro,)).fetchone()

    if not resultado:
        return False

    # NAO_VINCULADO não é terminal: será tentado novamente.
    return resultado["status"] in STATUS_TERMINAIS


def marcar_processado(
    registro,
    status,
    id_animal_final=None,
    observacao=""
):
    id_registro = registro.get(
        "_id_registro",
        registro.get("id_registro")
    )
    id_lote = registro.get(
        "_id_lote",
        registro.get("id_lote")
    )
    conjunto = registro.get(
        "_conjunto",
        registro.get("conjunto")
    )

    if id_registro is None or conjunto is None:
        raise ValueError(
            "Registro sem identificador técnico ou conjunto."
        )

    cursor.execute("""
        INSERT INTO controle_processamento (
            id_registro_bruto,
            id_lote,
            conjunto,
            versao_tratamento,
            status,
            id_animal_final,
            observacao,
            data_processamento
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(id_registro_bruto)
        DO UPDATE SET
            id_lote = excluded.id_lote,
            conjunto = excluded.conjunto,
            versao_tratamento = excluded.versao_tratamento,
            status = excluded.status,
            id_animal_final = excluded.id_animal_final,
            observacao = excluded.observacao,
            data_processamento = excluded.data_processamento
    """, (
        id_registro,
        id_lote,
        conjunto,
        VERSAO_TRATAMENTO,
        status,
        id_animal_final,
        observacao,
        datetime.now().isoformat(timespec="seconds")
    ))


def inserir_dinamico(tabela, dados, ignorar_conflito=False):
    """
    Não descarta silenciosamente colunas obrigatórias.
    Se o dicionário pedir uma coluna que não existe no banco,
    interrompe o processamento e informa a incompatibilidade.
    """
    colunas = colunas_tabela(tabela)
    ausentes = set(dados) - colunas

    if ausentes:
        raise RuntimeError(
            f"A tabela {tabela} não possui as colunas "
            f"esperadas: {sorted(ausentes)}"
        )

    if not dados:
        raise RuntimeError(
            f"Nenhum dado informado para inserir em {tabela}."
        )

    campos = list(dados)

    comando = (
        f"INSERT {'OR IGNORE ' if ignorar_conflito else ''}"
        f'INTO "{tabela}" '
        f"({', '.join('"' + campo + '"' for campo in campos)}) "
        f"VALUES ({', '.join(['?'] * len(campos))})"
    )

    cursor.execute(
        comando,
        [dados[campo] for campo in campos]
    )


def valor_mais_frequente(candidatos, campo, validador):
    valores = []

    for registro in candidatos:
        valor = registro.get(campo)

        try:
            valido = validador(valor)
        except (TypeError, ValueError):
            valido = False

        if valido:
            valores.append(valor)

    if not valores:
        return None

    contagem = Counter(valores)
    maior = max(contagem.values())

    empatados = {
        valor
        for valor, quantidade in contagem.items()
        if quantidade == maior
    }

    # Desempate pela primeira ocorrência no conjunto.
    for registro in candidatos:
        valor = registro.get(campo)
        try:
            if valor in empatados:
                return valor
        except TypeError:
            continue

    return next(iter(empatados))


def campo_textual_valido(valor):
    return bool(texto(valor))


def peso_valido(valor, especie):
    try:
        peso = float(str(valor).replace(",", "."))

        if especie == "Gato":
            return 1 <= peso <= 12

        if especie == "Cachorro":
            return 1 <= peso <= 60

        return False

    except (ValueError, TypeError):
        return False


def municipio_da_imagem(imagem):
    imagem = texto(imagem)

    for municipio in MUNICIPIOS_VALIDOS:
        prefixo = municipio.replace(" ", "_") + "_animal_"

        if imagem.startswith(prefixo):
            return municipio

    return None


def id_origem_registro(registro):
    meta = registro.get("_meta") or {}

    id_origem = texto(meta.get("id_animal_origem"))

    if not id_origem:
        id_origem = texto(registro.get("id_animal"))

    return id_origem or None


def buscar_mapeamento(id_origem):
    if not id_origem:
        return None

    linha = cursor.execute("""
        SELECT id_animal_final
        FROM mapeamento_animais
        WHERE id_animal_origem = ?
    """, (id_origem,)).fetchone()

    return linha["id_animal_final"] if linha else None


def salvar_mapeamento(id_origem, id_final, id_lote):
    """
    Um ID de origem sempre mantém o mesmo destino.
    Não substitui silenciosamente um mapeamento já existente.
    """
    existente = buscar_mapeamento(id_origem)

    if existente and existente != id_final:
        raise RuntimeError(
            f"Mapeamento conflitante para {id_origem}: "
            f"{existente} != {id_final}"
        )

    if not existente:
        cursor.execute("""
            INSERT INTO mapeamento_animais (
                id_animal_origem,
                id_animal_final,
                id_lote,
                data_mapeamento
            )
            VALUES (?, ?, ?, ?)
        """, (
            id_origem,
            id_final,
            id_lote,
            datetime.now().isoformat(timespec="seconds")
        ))


# 8. LEITURA DOS REGISTROS BRUTOS

brutos = cursor.execute("""
    SELECT
        id_registro,
        id_lote,
        conjunto,
        dados_json
    FROM registros_brutos
    ORDER BY id_registro
""").fetchall()

registros_por_conjunto = defaultdict(list)
quantidade_json_rejeitados = 0

for linha in brutos:
    conjunto = linha["conjunto"]

    if conjunto not in CONJUNTOS:
        continue

    if controle_existe(linha["id_registro"]):
        continue

    try:
        dados = json.loads(linha["dados_json"])

        if not isinstance(dados, dict):
            raise ValueError("O JSON não representa um objeto.")

        dados["_id_registro"] = linha["id_registro"]
        dados["_id_lote"] = linha["id_lote"]
        dados["_conjunto"] = conjunto

        registros_por_conjunto[conjunto].append(dados)

    except Exception as erro:
        registrar_problema(
            linha["id_registro"],
            linha["id_lote"],
            conjunto,
            None,
            "dados_json",
            linha["dados_json"],
            None,
            "json_invalido",
            f"Registro rejeitado: {erro}"
        )

        marcar_processado(
            {
                "id_registro": linha["id_registro"],
                "id_lote": linha["id_lote"],
                "conjunto": conjunto
            },
            "REJEITADO",
            observacao="JSON inválido ou estrutura incompatível."
        )

        quantidade_json_rejeitados += 1


# 9. AGRUPAMENTO DOS ANIMAIS POR ID DE ORIGEM

grupos_animais = defaultdict(list)

for animal in registros_por_conjunto["animais"]:
    id_origem = id_origem_registro(animal)

    if not id_origem:
        registrar_problema(
            animal["_id_registro"],
            animal["_id_lote"],
            "animais",
            None,
            "id_animal",
            animal.get("id_animal"),
            None,
            "identificador_irrecuperavel",
            "Registro sem identificador técnico recuperável."
        )

        marcar_processado(
            animal,
            "REJEITADO",
            observacao="Identificador do animal irrecuperável."
        )
        continue

    grupos_animais[id_origem].append(animal)


# 10. CONTADORES

mapa_animais = {}

quantidade_animais_inseridos = 0
quantidade_animais_atualizados = 0
quantidade_duplicidades_exatas = 0
quantidade_duplicidades_parciais = 0
quantidade_divergencias_peso = 0
quantidade_grupos_ja_processados = 0


# 11. TRATAMENTO E CONSOLIDAÇÃO DOS ANIMAIS

for id_origem, candidatos in grupos_animais.items():

    # ------------------------------------------------------------
    # 11.1. IDENTIFICAÇÃO DE DUPLICIDADES
    # ------------------------------------------------------------

    assinaturas = []

    for item in candidatos:
        dados_assinatura = {
            chave: valor
            for chave, valor in item.items()
            if chave not in {
                "_id_registro",
                "_id_lote",
                "_conjunto"
            }
        }

        assinaturas.append(serializar(dados_assinatura))

    contagem_assinaturas = Counter(assinaturas)

    quantidade_duplicidades_exatas += sum(
        quantidade - 1
        for quantidade in contagem_assinaturas.values()
        if quantidade > 1
    )

    quantidade_duplicidades_parciais += sum(
        1
        for item in candidatos
        if "duplicidade_parcial" in (
            (item.get("_meta") or {}).get(
                "problemas_introduzidos", []
            )
        )
    )

    # ------------------------------------------------------------
    # 11.2. CONSOLIDAÇÃO DOS CAMPOS
    # ------------------------------------------------------------

    especie = valor_mais_frequente(
        candidatos,
        "especie",
        lambda v: v in ESPECIES_VALIDAS
    ) or "Não informado"

    nome = valor_mais_frequente(
        candidatos, "nome", campo_textual_valido
    )

    raca = valor_mais_frequente(
        candidatos, "raca", campo_textual_valido
    )

    sexo = valor_mais_frequente(
        candidatos, "sexo", lambda v: v in SEXOS_VALIDOS
    )

    municipio = valor_mais_frequente(
        candidatos,
        "municipio",
        lambda v: v in MUNICIPIOS_VALIDOS
    )

    if not municipio:
        municipios_imagem = [
            municipio_da_imagem(item.get("imagem"))
            for item in candidatos
        ]
        municipios_imagem = [
            item for item in municipios_imagem if item
        ]
        municipio = (
            Counter(municipios_imagem).most_common(1)[0][0]
            if municipios_imagem else None
        )

    peso = valor_mais_frequente(
        candidatos,
        "peso_kg",
        lambda v: peso_valido(v, especie)
    )

    pesos_validos = []
    for item in candidatos:
        if peso_valido(item.get("peso_kg"), especie):
            pesos_validos.append(
                float(str(item["peso_kg"]).replace(",", "."))
            )

    revisar_peso = ""

    if len(set(pesos_validos)) > 1:
        revisar_peso = "Verificar peso"
        quantidade_divergencias_peso += 1
    elif peso is None:
        revisar_peso = "Verificar peso"

    status = valor_mais_frequente(
        candidatos, "status", lambda v: v in STATUS_VALIDOS
    )

    situacao_vacinacao = valor_mais_frequente(
        candidatos,
        "situacao_vacinacao",
        lambda v: v in {"Em dia", "Pendente", "Parcial"}
    )

    data_acolhimento = valor_mais_frequente(
        candidatos,
        "data_acolhimento",
        lambda v: data_valida(v) is not None
    )
    data_acolhimento = data_valida(data_acolhimento)

    motivo_acolhimento = valor_mais_frequente(
        candidatos,
        "motivo_acolhimento",
        lambda v: v in MOTIVOS_ACOLHIMENTO
    )

    origem_acolhimento = valor_mais_frequente(
        candidatos,
        "origem_acolhimento",
        lambda v: v in ORIGENS_ACOLHIMENTO
    )

    imagem = valor_mais_frequente(
        candidatos, "imagem", campo_textual_valido
    )

    vacinas_faltantes = valor_mais_frequente(
        candidatos,
        "vacinas_faltantes",
        lambda v: v is not None
    )

    # ------------------------------------------------------------
    # 11.3. IDENTIFICADOR FINAL E PROTEÇÃO DOS DADOS MANUAIS
    # ------------------------------------------------------------

    mapeamento_existente = buscar_mapeamento(id_origem)

    if mapeamento_existente:
        id_final = mapeamento_existente

        destino = cursor.execute("""
            SELECT origem
            FROM animais
            WHERE id_animal = ?
        """, (id_final,)).fetchone()

        if not destino:
            raise RuntimeError(
                f"O mapeamento {id_origem} -> {id_final} "
                "não possui animal correspondente. "
                "A execução foi interrompida para evitar "
                "recriação ou troca indevida do vínculo."
            )

        if origem_manual(destino["origem"]):
            raise RuntimeError(
                f"O mapeamento de {id_origem} aponta para "
                f"o cadastro manual {id_final}. "
                "Nenhum dado manual foi alterado."
            )

    else:
        # Primeiro tenta manter o ID original se estiver livre
        # ou se já pertencer a um animal não manual.
        id_final = id_origem

        conflito = cursor.execute("""
            SELECT origem
            FROM animais
            WHERE id_animal = ?
        """, (id_final,)).fetchone()

        if conflito and origem_manual(conflito["origem"]):
            base = "SINT-" + id_origem
            id_final = base
            sufixo = 2

            while True:
                conflito_sintetico = cursor.execute("""
                    SELECT origem
                    FROM animais
                    WHERE id_animal = ?
                """, (id_final,)).fetchone()

                if not conflito_sintetico:
                    break

                if not origem_manual(conflito_sintetico["origem"]):
                    break

                id_final = f"{base}-{sufixo}"
                sufixo += 1

        salvar_mapeamento(
            id_origem,
            id_final,
            candidatos[0]["_id_lote"]
        )

    mapa_animais[id_origem] = id_final

    # Verificação adicional: nunca atualizar cadastro manual.
    existente_final = cursor.execute("""
        SELECT origem
        FROM animais
        WHERE id_animal = ?
    """, (id_final,)).fetchone()

    if existente_final and origem_manual(existente_final["origem"]):
        raise RuntimeError(
            f"Proteção de integridade: {id_final} pertence "
            "a um cadastro manual. A execução foi interrompida."
        )

    # ------------------------------------------------------------
    # 11.4. EVENTOS TERMINAIS JÁ PRESENTES NESTE LOTE/EXECUÇÃO
    # ------------------------------------------------------------

    possui_evento_terminal = any(
        id_origem_registro(evento) == id_origem
        and texto(evento.get("tipo_evento")) in TIPOS_EVENTO
        for evento in registros_por_conjunto["obitos_eutanasias"]
    )

    if possui_evento_terminal:
        status = "Inativo"

    # ------------------------------------------------------------
    # 11.5. PADRONIZAÇÃO DE VALORES AUSENTES
    # ------------------------------------------------------------

    nome = nome or "Não informado"
    raca = raca or "Não informada"
    sexo = sexo or "Não informado"
    municipio = municipio or "Não informado"
    status = status or "Ativo"
    situacao_vacinacao = situacao_vacinacao or "Não informado"
    motivo_acolhimento = motivo_acolhimento or "Não informado"
    origem_acolhimento = origem_acolhimento or "Não informado"

    # ------------------------------------------------------------
    # 11.6. REGISTRO DOS PROBLEMAS MARCADOS PELO GERADOR
    # ------------------------------------------------------------

    for item in candidatos:
        meta = item.get("_meta") or {}
        problemas_meta = meta.get("problemas_introduzidos") or []

        for problema in problemas_meta:
            registrar_problema(
                item["_id_registro"],
                item["_id_lote"],
                "animais",
                id_final,
                "registro",
                problema,
                "Tratado ou encaminhado para revisão",
                problema,
                "Problema identificado pela marcação do gerador."
            )

    # ------------------------------------------------------------
    # 11.7. REGISTRO DAS CORREÇÕES
    # ------------------------------------------------------------

    campos_verificados = [
        ("nome", nome),
        ("raca", raca),
        ("especie", especie),
        ("sexo", sexo),
        ("municipio", municipio),
        ("peso_kg", peso),
        ("status", status),
        ("situacao_vacinacao", situacao_vacinacao),
        ("data_acolhimento", data_acolhimento),
        ("motivo_acolhimento", motivo_acolhimento),
        ("origem_acolhimento", origem_acolhimento)
    ]

    campos_que_exigem_validacao = {
        "peso_kg", "especie", "municipio", "sexo", "status",
        "situacao_vacinacao", "data_acolhimento",
        "motivo_acolhimento", "origem_acolhimento"
    }

    for item in candidatos:
        for campo, corrigido in campos_verificados:
            original = item.get(campo)

            if original == corrigido:
                continue

            if (
                original is None
                or original == ""
                or campo in campos_que_exigem_validacao
            ):
                registrar_problema(
                    item["_id_registro"],
                    item["_id_lote"],
                    "animais",
                    id_final,
                    campo,
                    original,
                    corrigido,
                    "valor_ausente_ou_invalido",
                    "Valor padronizado ou encaminhado para revisão."
                )

    # ------------------------------------------------------------
    # 11.8. INSERÇÃO OU ATUALIZAÇÃO DO ANIMAL
    # ------------------------------------------------------------

    dados_animal = {
        "id_animal": id_final,
        "nome": nome,
        "especie": especie,
        "raca": raca,
        "sexo": sexo,
        "municipio": municipio,
        "peso_kg": peso,
        "status": status,
        "vacinas_faltantes": texto(vacinas_faltantes),
        "situacao_vacinacao": situacao_vacinacao,
        "imagem": imagem,
        "revisao_peso": revisar_peso,
        "data_cadastro": datetime.now().isoformat(timespec="seconds"),
        "id_lote": candidatos[0]["_id_lote"],
        "origem": "Sintético",
        "data_acolhimento": data_acolhimento,
        "motivo_acolhimento": motivo_acolhimento,
        "origem_acolhimento": origem_acolhimento
    }

    if existente_final:
        # Atualiza somente cadastro não manual.
        campos_atualizaveis = {
            campo: valor
            for campo, valor in dados_animal.items()
            if campo not in {"id_animal", "origem"}
        }

        cursor.execute(
            "UPDATE animais SET "
            + ", ".join(
                f'"{campo}" = ?'
                for campo in campos_atualizaveis
            )
            + " WHERE id_animal = ? "
              "AND LOWER(COALESCE(origem, '')) <> 'manual'",
            list(campos_atualizaveis.values()) + [id_final]
        )

        if cursor.rowcount > 0:
            quantidade_animais_atualizados += 1

    else:
        inserir_dinamico("animais", dados_animal)
        quantidade_animais_inseridos += 1

    # ------------------------------------------------------------
    # 11.9. MARCAÇÃO DOS REGISTROS BRUTOS
    # ------------------------------------------------------------

    for item in candidatos:
        marcar_processado(
            item,
            "TRATADO",
            id_final,
            "Animal consolidado na tabela definitiva."
        )


# 12. RESOLUÇÃO SEGURA DOS RELACIONAMENTOS

def resolver_animal(registro):
    """
    Prioridade:
    1. Mapeamento persistente de ID sintético.
    2. ID direto somente se existir e não for Manual.
    3. Nunca retorna um ID inexistente como se fosse vínculo.
    """
    id_origem = id_origem_registro(registro)

    if not id_origem:
        return None

    if id_origem in mapa_animais:
        id_final = mapa_animais[id_origem]
    else:
        id_final = buscar_mapeamento(id_origem)

    if id_final:
        linha = cursor.execute("""
            SELECT id_animal, origem
            FROM animais
            WHERE id_animal = ?
        """, (id_final,)).fetchone()

        if not linha:
            return None

        if origem_manual(linha["origem"]):
            return None

        return linha["id_animal"]

    # Compatibilidade com animais sintéticos antigos que ainda
    # não possuem mapeamento persistido.
    linha = cursor.execute("""
        SELECT id_animal, origem
        FROM animais
        WHERE id_animal = ?
    """, (id_origem,)).fetchone()

    if not linha:
        return None

    if origem_manual(linha["origem"]):
        # Não vincula automaticamente um registro sintético
        # a um cadastro manual com ID coincidente.
        return None

    salvar_mapeamento(
        id_origem,
        linha["id_animal"],
        None
    )

    return linha["id_animal"]


def registro_tem_id_origem(registro):
    return bool(id_origem_registro(registro))


# 13. TRATAMENTO DAS VACINAÇÕES

quantidade_vacinacoes = 0
quantidade_vacinacoes_rejeitadas = 0
quantidade_vacinacoes_pendentes = 0

for registro in registros_por_conjunto["vacinacoes"]:
    id_bruto = registro["_id_registro"]

    if controle_existe(id_bruto):
        continue

    id_animal = resolver_animal(registro)

    if not id_animal:
        status_pendente = (
            "NAO_VINCULADO"
            if registro_tem_id_origem(registro)
            else "REJEITADO"
        )

        marcar_processado(
            registro,
            status_pendente,
            observacao=(
                "Animal ainda não localizado; será tentado novamente."
                if status_pendente == "NAO_VINCULADO"
                else "Vacinação sem identificador de animal recuperável."
            )
        )

        if status_pendente == "NAO_VINCULADO":
            quantidade_vacinacoes_pendentes += 1
        continue

    animal_info = cursor.execute("""
        SELECT especie
        FROM animais
        WHERE id_animal = ?
    """, (id_animal,)).fetchone()

    if not animal_info:
        marcar_processado(
            registro,
            "NAO_VINCULADO",
            observacao="Animal não encontrado; nova tentativa futura."
        )
        quantidade_vacinacoes_pendentes += 1
        continue

    especie_animal = animal_info["especie"]
    nome_vacina = texto(registro.get("nome_vacina"))

    if nome_vacina not in VACINAS_POR_ESPECIE.get(
        especie_animal, set()
    ):
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "vacinacoes",
            id_animal,
            "nome_vacina",
            registro.get("nome_vacina"),
            "Não informado",
            "vacina_incompativel_ou_invalida",
            "Vacina incompatível com a espécie ou não reconhecida."
        )
        nome_vacina = "Não informado"
        quantidade_vacinacoes_rejeitadas += 1

    data_aplicacao = data_valida(registro.get("data_aplicacao"))

    if not data_aplicacao:
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "vacinacoes",
            id_animal,
            "data_aplicacao",
            registro.get("data_aplicacao"),
            None,
            "data_invalida_ou_futura",
            "Data removida para revisão."
        )

    proxima_dose = data_valida(registro.get("proxima_dose"))

    if (
        proxima_dose
        and data_aplicacao
        and proxima_dose < data_aplicacao
    ):
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "vacinacoes",
            id_animal,
            "proxima_dose",
            registro.get("proxima_dose"),
            None,
            "proxima_dose_anterior",
            "Data removida para revisão."
        )
        proxima_dose = None

    situacao = texto(registro.get("situacao"))

    if situacao not in SITUACOES_VACINA:
        situacao = "Pendente"
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "vacinacoes",
            id_animal,
            "situacao",
            registro.get("situacao"),
            situacao,
            "situacao_invalida",
            "Situação padronizada."
        )

    inserir_dinamico(
        "vacinacoes",
        {
            "id_animal": id_animal,
            "nome_vacina": nome_vacina,
            "data_aplicacao": data_aplicacao,
            "proxima_dose": proxima_dose,
            "situacao": situacao,
            "observacoes": texto(registro.get("observacoes"))
        }
    )

    marcar_processado(registro, "TRATADO", id_animal)
    quantidade_vacinacoes += 1


# 14. TRATAMENTO DOS DIAGNÓSTICOS

quantidade_diagnosticos = 0
quantidade_diagnosticos_nao_vinculados = 0

for registro in registros_por_conjunto["diagnosticos"]:
    id_bruto = registro["_id_registro"]

    if controle_existe(id_bruto):
        continue

    id_animal = resolver_animal(registro)

    if not id_animal:
        status_pendente = (
            "NAO_VINCULADO"
            if registro_tem_id_origem(registro)
            else "REJEITADO"
        )

        marcar_processado(
            registro,
            status_pendente,
            observacao=(
                "Animal não localizado; será tentado novamente."
                if status_pendente == "NAO_VINCULADO"
                else "Diagnóstico sem identificador de animal recuperável."
            )
        )

        if status_pendente == "NAO_VINCULADO":
            quantidade_diagnosticos_nao_vinculados += 1
        continue

    diagnostico = texto(registro.get("diagnostico"))

    if not diagnostico:
        diagnostico = "Não informado"
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "diagnosticos",
            id_animal,
            "diagnostico",
            registro.get("diagnostico"),
            diagnostico,
            "diagnostico_ausente",
            "Campo preenchido com indicação de ausência."
        )

    data_diagnostico = data_valida(
        registro.get("data_diagnostico")
    )

    if not data_diagnostico:
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "diagnosticos",
            id_animal,
            "data_diagnostico",
            registro.get("data_diagnostico"),
            None,
            "data_invalida_ou_futura",
            "Data removida para revisão."
        )

    situacao = texto(registro.get("situacao"))

    if situacao not in SITUACOES_DIAGNOSTICO:
        situacao = "Em acompanhamento"
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "diagnosticos",
            id_animal,
            "situacao",
            registro.get("situacao"),
            situacao,
            "situacao_invalida",
            "Situação padronizada."
        )

    inserir_dinamico(
        "diagnosticos",
        {
            "id_animal": id_animal,
            "data_diagnostico": data_diagnostico,
            "diagnostico": diagnostico,
            "descricao": texto(registro.get("descricao")),
            "situacao": situacao,
            "origem": "Sintético"
        }
    )

    marcar_processado(registro, "TRATADO", id_animal)
    quantidade_diagnosticos += 1


# 15. TRATAMENTO DE MAUS-TRATOS

quantidade_maus_tratos = 0
quantidade_maus_tratos_nao_vinculados = 0

for registro in registros_por_conjunto["maus_tratos"]:
    id_bruto = registro["_id_registro"]

    if controle_existe(id_bruto):
        continue

    id_origem = id_origem_registro(registro)
    id_animal = resolver_animal(registro)

    # Se há ID de origem, mas o animal ainda não chegou,
    # não grava como ocorrência definitivamente sem vínculo.
    if id_origem and not id_animal:
        marcar_processado(
            registro,
            "NAO_VINCULADO",
            observacao=(
                "Ocorrência possui ID de animal ainda não localizado; "
                "será tentada novamente."
            )
        )
        quantidade_maus_tratos_nao_vinculados += 1
        continue

    # Sem ID de origem, a ocorrência pode ser legítima e independente.
    if not id_origem:
        id_animal = None
        quantidade_maus_tratos_nao_vinculados += 1

    municipio = registro.get("municipio")

    if municipio not in MUNICIPIOS_VALIDOS:
        municipio = "Não informado"
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "maus_tratos",
            id_animal,
            "municipio",
            registro.get("municipio"),
            municipio,
            "municipio_invalido",
            "Município não recuperável com segurança."
        )

    tipo_ocorrencia = texto(registro.get("tipo_ocorrencia"))

    if not tipo_ocorrencia:
        tipo_ocorrencia = "Não informado"
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "maus_tratos",
            id_animal,
            "tipo_ocorrencia",
            registro.get("tipo_ocorrencia"),
            tipo_ocorrencia,
            "tipo_ocorrencia_ausente",
            "Campo preenchido com indicação de ausência."
        )

    data_ocorrencia = data_valida(
        registro.get("data_ocorrencia")
    )

    if not data_ocorrencia:
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "maus_tratos",
            id_animal,
            "data_ocorrencia",
            registro.get("data_ocorrencia"),
            None,
            "data_invalida_ou_futura",
            "Data removida para revisão."
        )

    situacao = texto(registro.get("situacao"))

    if situacao not in SITUACOES_OCORRENCIA:
        situacao = "Em investigação"
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "maus_tratos",
            id_animal,
            "situacao",
            registro.get("situacao"),
            situacao,
            "situacao_invalida",
            "Situação padronizada."
        )

    inserir_dinamico(
        "maus_tratos",
        {
            "id_animal": id_animal,
            "municipio": municipio,
            "data_ocorrencia": data_ocorrencia,
            "tipo_ocorrencia": tipo_ocorrencia,
            "descricao": texto(registro.get("descricao")),
            "situacao": situacao,
            "origem": "Sintético"
        }
    )

    marcar_processado(
        registro,
        "TRATADO",
        id_animal,
        (
            "Ocorrência registrada com vínculo animal."
            if id_animal
            else "Ocorrência independente, sem animal identificado."
        )
    )
    quantidade_maus_tratos += 1


# 16. TRATAMENTO DE ÓBITOS E EUTANÁSIA

quantidade_eventos = 0
quantidade_eventos_nao_vinculados = 0
quantidade_animais_inativados = 0

for registro in registros_por_conjunto["obitos_eutanasias"]:
    id_bruto = registro["_id_registro"]

    if controle_existe(id_bruto):
        continue

    id_origem = id_origem_registro(registro)
    id_animal = resolver_animal(registro)

    # ID presente e animal ainda inexistente: aguardar lote futuro.
    if id_origem and not id_animal:
        marcar_processado(
            registro,
            "NAO_VINCULADO",
            observacao=(
                "Evento possui ID de animal ainda não localizado; "
                "será tentado novamente."
            )
        )
        quantidade_eventos_nao_vinculados += 1
        continue

    # Sem ID de origem, o evento é preservado sem vínculo.
    if not id_origem:
        id_animal = None
        quantidade_eventos_nao_vinculados += 1

    tipo_evento = texto(registro.get("tipo_evento"))
    evento_valido = tipo_evento in TIPOS_EVENTO

    if not evento_valido:
        tipo_evento = "Não classificado"
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "obitos_eutanasias",
            id_animal,
            "tipo_evento",
            registro.get("tipo_evento"),
            tipo_evento,
            "tipo_evento_invalido",
            "Evento preservado, mas encaminhado para classificação."
        )

    data_evento_original = registro.get("data_evento")
    data_evento = data_valida(data_evento_original)

    motivo = texto(registro.get("motivo"))

    if not motivo:
        motivo = "Não informado"
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "obitos_eutanasias",
            id_animal,
            "motivo",
            registro.get("motivo"),
            motivo,
            "motivo_ausente",
            "Campo preenchido com indicação de ausência."
        )

    municipio = registro.get("municipio")

    if municipio not in MUNICIPIOS_VALIDOS:
        municipio = "Não informado"
        registrar_problema(
            id_bruto,
            registro["_id_lote"],
            "obitos_eutanasias",
            id_animal,
            "municipio",
            registro.get("municipio"),
            municipio,
            "municipio_invalido",
            "Município não recuperável com segurança."
        )

    # Não inventa uma nova data para corrigir evento anterior
    # ao acolhimento. Remove a data tratada e registra a divergência.
    if id_animal and data_evento:
        acolhimento = cursor.execute("""
            SELECT data_acolhimento
            FROM animais
            WHERE id_animal = ?
        """, (id_animal,)).fetchone()

        data_acolhimento_final = (
            data_valida(acolhimento["data_acolhimento"])
            if acolhimento else None
        )

        if (
            data_acolhimento_final
            and data_evento < data_acolhimento_final
        ):
            registrar_problema(
                id_bruto,
                registro["_id_lote"],
                "obitos_eutanasias",
                id_animal,
                "data_evento",
                data_evento,
                None,
                "evento_anterior_ao_acolhimento",
                "Data removida para revisão; a data original "
                "permanece preservada no registro bruto."
            )
            data_evento = None

    if not data_evento and data_evento_original:
        # Evita duplicar a ocorrência de problema já registrada
        # no caso específico de data anterior ao acolhimento.
        if not (
            id_animal
            and data_valida(data_evento_original)
            and cursor.execute("""
                SELECT 1
                FROM qualidade_dados
                WHERE id_registro_bruto = ?
                  AND tipo_problema = 'evento_anterior_ao_acolhimento'
                LIMIT 1
            """, (id_bruto,)).fetchone()
        ):
            registrar_problema(
                id_bruto,
                registro["_id_lote"],
                "obitos_eutanasias",
                id_animal,
                "data_evento",
                data_evento_original,
                None,
                "data_invalida_ou_futura",
                "Data removida para revisão."
            )

    inserir_dinamico(
        "obitos_eutanasias",
        {
            "id_animal": id_animal,
            "tipo_evento": tipo_evento,
            "data_evento": data_evento,
            "municipio": municipio,
            "motivo": motivo,
            "observacoes": texto(registro.get("observacoes")),
            "origem": "Sintético"
        }
    )

    if evento_valido and id_animal:
        resultado = cursor.execute("""
            UPDATE animais
            SET status = 'Inativo'
            WHERE id_animal = ?
              AND LOWER(COALESCE(origem, '')) <> 'manual'
              AND COALESCE(status, '') <> 'Inativo'
        """, (id_animal,))

        if resultado.rowcount > 0:
            quantidade_animais_inativados += 1
            registrar_problema(
                id_bruto,
                registro["_id_lote"],
                "obitos_eutanasias",
                id_animal,
                "status",
                "Ativo ou não informado",
                "Inativo",
                "animal_ativo_apos_obito_ou_eutanasia",
                "Status atualizado após evento terminal válido."
            )

    marcar_processado(
        registro,
        "TRATADO",
        id_animal,
        (
            "Evento registrado com vínculo animal."
            if id_animal
            else "Evento independente, sem animal identificado."
        )
    )
    quantidade_eventos += 1


# 17. ATUALIZAÇÃO DA SITUAÇÃO VACINAL NÃO INFORMADA

animais_para_atualizar = cursor.execute("""
    SELECT id_animal
    FROM animais
    WHERE LOWER(COALESCE(origem, '')) <> 'manual'
      AND situacao_vacinacao = 'Não informado'
""").fetchall()

for linha in animais_para_atualizar:
    id_final = linha["id_animal"]

    aplicadas = cursor.execute("""
        SELECT COUNT(*)
        FROM vacinacoes
        WHERE id_animal = ?
          AND situacao = 'Aplicada'
          AND nome_vacina <> 'Não informado'
    """, (id_final,)).fetchone()[0]

    nova_situacao = "Parcial" if aplicadas > 0 else "Pendente"

    cursor.execute("""
        UPDATE animais
        SET situacao_vacinacao = ?
        WHERE id_animal = ?
          AND LOWER(COALESCE(origem, '')) <> 'manual'
          AND situacao_vacinacao = 'Não informado'
    """, (nova_situacao, id_final))


# 18. RESUMO DA QUALIDADE DOS DADOS

quantidade_problemas_acumulados = cursor.execute("""
    SELECT COUNT(*)
    FROM qualidade_dados
""").fetchone()[0]

resumo = {
    "animais_inseridos": quantidade_animais_inseridos,
    "animais_atualizados": quantidade_animais_atualizados,
    "grupos_animais_processados": len(grupos_animais),
    "grupos_ja_processados": quantidade_grupos_ja_processados,
    "duplicidades_exatas": quantidade_duplicidades_exatas,
    "duplicidades_parciais": quantidade_duplicidades_parciais,
    "divergencias_de_peso": quantidade_divergencias_peso,
    "vacinacoes_inseridas": quantidade_vacinacoes,
    "vacinacoes_com_problemas": quantidade_vacinacoes_rejeitadas,
    "vacinacoes_aguardando_vinculo": quantidade_vacinacoes_pendentes,
    "diagnosticos_inseridos": quantidade_diagnosticos,
    "diagnosticos_aguardando_vinculo": quantidade_diagnosticos_nao_vinculados,
    "maus_tratos_inseridos": quantidade_maus_tratos,
    "maus_tratos_aguardando_vinculo": quantidade_maus_tratos_nao_vinculados,
    "eventos_inseridos": quantidade_eventos,
    "eventos_aguardando_vinculo": quantidade_eventos_nao_vinculados,
    "animais_inativados_por_eventos": quantidade_animais_inativados,
    "json_rejeitados_nesta_execucao": quantidade_json_rejeitados,
    "problemas_acumulados_no_banco": quantidade_problemas_acumulados
}


# 19. REGISTRO DE AUDITORIA

dados_auditoria = {
    "entidade": "Tratamento de dados",
    "id_registro": None,
    "operacao": "TRATAMENTO",
    "descricao": json.dumps(resumo, ensure_ascii=False),
    "data_operacao": datetime.now().isoformat(timespec="seconds")
}

inserir_dinamico("auditoria", dados_auditoria)


# 20. CONFERÊNCIA DOS CONTROLES

controles = cursor.execute("""
    SELECT
        conjunto,
        status,
        COUNT(*) AS quantidade
    FROM controle_processamento
    GROUP BY conjunto, status
    ORDER BY conjunto, status
""").fetchall()

brutos_por_conjunto = cursor.execute("""
    SELECT conjunto, COUNT(*) AS quantidade
    FROM registros_brutos
    GROUP BY conjunto
""").fetchall()

controles_por_conjunto = cursor.execute("""
    SELECT conjunto, COUNT(*) AS quantidade
    FROM controle_processamento
    GROUP BY conjunto
""").fetchall()

totais_brutos = {
    linha["conjunto"]: linha["quantidade"]
    for linha in brutos_por_conjunto
}

totais_controles = {
    linha["conjunto"]: linha["quantidade"]
    for linha in controles_por_conjunto
}

conjuntos_sem_controle = {}

for conjunto in CONJUNTOS:
    conjuntos_sem_controle[conjunto] = (
        totais_brutos.get(conjunto, 0)
        - totais_controles.get(conjunto, 0)
    )

registros_pendentes = cursor.execute("""
    SELECT conjunto, COUNT(*) AS quantidade
    FROM controle_processamento
    WHERE status = 'NAO_VINCULADO'
    GROUP BY conjunto
    ORDER BY conjunto
""").fetchall()

animais_sem_correspondencia = cursor.execute("""
    SELECT COUNT(*)
    FROM controle_processamento c
    LEFT JOIN animais a
        ON a.id_animal = c.id_animal_final
    WHERE c.conjunto = 'animais'
      AND c.status = 'TRATADO'
      AND (
          c.id_animal_final IS NULL
          OR a.id_animal IS NULL
      )
""").fetchone()[0]

mapeamentos_sem_animal = cursor.execute("""
    SELECT COUNT(*)
    FROM mapeamento_animais m
    LEFT JOIN animais a
        ON a.id_animal = m.id_animal_final
    WHERE a.id_animal IS NULL
""").fetchone()[0]

mapeamentos_para_manuais = cursor.execute("""
    SELECT COUNT(*)
    FROM mapeamento_animais m
    JOIN animais a
        ON a.id_animal = m.id_animal_final
    WHERE LOWER(COALESCE(a.origem, '')) = 'manual'
""").fetchone()[0]


# 21. COMMIT

conexao.commit()


# 22. RELATÓRIO FINAL

print("\n" + "=" * 68)
print("TRATAMENTO CONCLUÍDO — VERSÃO 1.4")
print("=" * 68)

print("\nRESUMO DA EXECUÇÃO")

for campo, valor in resumo.items():
    print(f"{campo.replace('_', ' ').capitalize()}: {valor}")

print("\nCONTROLE DE PROCESSAMENTO")

for linha in controles:
    print(
        f"{linha['conjunto']} — "
        f"{linha['status']}: {linha['quantidade']}"
    )

print("\nREGISTROS SEM CONTROLE")

for conjunto, quantidade in conjuntos_sem_controle.items():
    print(f"{conjunto}: {quantidade}")

print("\nREGISTROS AGUARDANDO VÍNCULO")

if registros_pendentes:
    for linha in registros_pendentes:
        print(
            f"{linha['conjunto']}: {linha['quantidade']}"
        )
else:
    print("Nenhum registro pendente.")

print("\nVERIFICAÇÃO DOS RELACIONAMENTOS")

print(
    "Animais tratados sem correspondência:",
    animais_sem_correspondencia
)
print(
    "Mapeamentos sem animal correspondente:",
    mapeamentos_sem_animal
)
print(
    "Mapeamentos apontando para cadastros manuais:",
    mapeamentos_para_manuais
)

if (
    all(v == 0 for v in conjuntos_sem_controle.values())
    and animais_sem_correspondencia == 0
    and mapeamentos_sem_animal == 0
    and mapeamentos_para_manuais == 0
):
    print("\nVERIFICAÇÃO ESTRUTURAL CONCLUÍDA.")
else:
    print(
        "\nATENÇÃO: existem registros sem controle, "
        "vínculos pendentes ou inconsistências de mapeamento."
    )

print("\nREGISTROS BRUTOS PRESERVADOS.")
print("DADOS MANUAIS PROTEGIDOS.")
print("CONTROLES DE PROCESSAMENTO ATUALIZADOS.")
print("HISTÓRICO DE QUALIDADE REGISTRADO.")
print("MAPEAMENTOS DOS ANIMAIS PRESERVADOS.")
print("=" * 68)

conexao.close()